In [1]:
import pandas as pd
import requests

print("Notebook funktioniert!")
print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)

Notebook funktioniert!
Pandas: 3.0.6
Requests: 2.34.2


In [15]:
import requests
from datetime import datetime, timezone
from uuid import uuid4

URL = "https://api.opentransportdata.swiss/ojp20"

STOP_ID = "ch:1:sloid:3000"
STOP_NAME = "Zürich HB"

now_utc = datetime.now(timezone.utc)
timestamp = now_utc.isoformat(timespec="milliseconds").replace("+00:00", "Z")

message_id = f"zhaw-pilot-{uuid4()}"

xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xsi:schemaLocation="http://www.vdv.de/ojp"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:ServiceRequestContext>
                <siri:Language>de</siri:Language>
            </siri:ServiceRequestContext>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPStopEventRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>

                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <Location>

                    <PlaceRef>

                        <siri:StopPointRef>{STOP_ID}</siri:StopPointRef>

                        <Name>
                            <Text>{STOP_NAME}</Text>
                        </Name>

                    </PlaceRef>

                    <DepArrTime>{timestamp}</DepArrTime>

                </Location>

                <Params>

                    <NumberOfResults>30</NumberOfResults>

                    <StopEventType>departure</StopEventType>

                    <IncludePreviousCalls>false</IncludePreviousCalls>

                    <IncludeOnwardCalls>false</IncludeOnwardCalls>

                    <UseRealtimeData>full</UseRealtimeData>

                </Params>

            </OJPStopEventRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

headers = {
    "Content-Type": "application/xml",
    "Authorization": f"Bearer {TOKEN}"
}

response = requests.post(
    URL,
    headers=headers,
    data=xml_request.encode("utf-8"),
    timeout=30
)

print("HTTP Status:", response.status_code)
print()
print("Antwort der API:")
print(response.text[:3000])

HTTP Status: 200

Antwort der API:
<?xml version="1.0" encoding="utf-8"?><OJP xmlns:siri="http://www.siri.org.uk/siri" version="2.0" xmlns="http://www.vdv.de/ojp"><OJPResponse><siri:ServiceDelivery><siri:ResponseTimestamp>2026-09-19T12:23:29.335577+02:00</siri:ResponseTimestamp><siri:ProducerRef>MENTZ-prod-asg_1.1.17.555_i-0d9ca8b0d9</siri:ProducerRef><siri:ResponseMessageIdentifier>00-8273ce8ace8ddf25eaa45</siri:ResponseMessageIdentifier><OJPStopEventDelivery><siri:ResponseTimestamp>2026-09-19T12:23:29.3348078+02:00</siri:ResponseTimestamp><siri:RequestMessageRef>zhaw-pilot-b71bf8a0-1bad-4838-a253-f89dd2544924</siri:RequestMessageRef><siri:DefaultLanguage>de</siri:DefaultLanguage><CalcTime>50</CalcTime><StopEventResponseContext><Places><Place><StopPlace><StopPlaceRef>ch:1:sloid:3000</StopPlaceRef><StopPlaceName><Text xml:lang="de">Zürich HB</Text></StopPlaceName><PrivateCode><System>EFA</System><Value>108276:0:43_x002F_44</Value></PrivateCode><TopographicPlaceRef>23026261:27</Topograp

In [16]:
import xml.etree.ElementTree as ET

# XML-Antwort in eine durchsuchbare Struktur umwandeln
root = ET.fromstring(response.content)

# XML namespaces
ns = {
    "ojp": "http://www.vdv.de/ojp",
    "siri": "http://www.siri.org.uk/siri"
}

# Alle gefundenen Verbindungen / Stop Events suchen
results = root.findall(".//ojp:StopEventResult", ns)

print("Gefundene Verbindungen:", len(results))

# Erste Verbindung vollständig anzeigen
if len(results) > 0:
    first_result = ET.tostring(
        results[0],
        encoding="unicode"
    )

    print("\nErste Verbindung:")
    print(first_result[:5000])
else:
    print("Keine StopEventResult gefunden.")

Gefundene Verbindungen: 30

Erste Verbindung:
<ns0:StopEventResult xmlns:ns0="http://www.vdv.de/ojp" xmlns:ns1="http://www.siri.org.uk/siri"><ns0:Id>c2ac3c68-5ae3-47a2-bd5f-7ac7b6ca4e92</ns0:Id><ns0:StopEvent><ns0:ThisCall><ns0:CallAtStop><ns1:StopPointRef>ch:1:sloid:3000:503:43</ns1:StopPointRef><ns0:StopPointName><ns0:Text xml:lang="de">Zürich HB</ns0:Text></ns0:StopPointName><ns0:NameSuffix><ns0:Text xml:lang="de">PLATFORM_ACCESS_WITHOUT_ASSISTANCE</ns0:Text></ns0:NameSuffix><ns0:PlannedQuay><ns0:Text xml:lang="de">43/44</ns0:Text></ns0:PlannedQuay><ns0:EstimatedQuay><ns0:Text xml:lang="de">44</ns0:Text></ns0:EstimatedQuay><ns0:ServiceDeparture><ns0:TimetabledTime>2026-09-19T10:24:00Z</ns0:TimetabledTime><ns0:EstimatedTime>2026-09-19T10:24:18Z</ns0:EstimatedTime></ns0:ServiceDeparture><ns0:Order>14</ns0:Order><ns1:ExpectedDepartureOccupancy><ns1:FareClass>firstClass</ns1:FareClass><ns1:OccupancyLevel>manySeatsAvailable</ns1:OccupancyLevel></ns1:ExpectedDepartureOccupancy><ns1:Expect

In [17]:
import pandas as pd
import xml.etree.ElementTree as ET

# XML erneut parsen
root = ET.fromstring(response.content)

# Namespaces
ns = {
    "ojp": "http://www.vdv.de/ojp",
    "siri": "http://www.siri.org.uk/siri"
}

# Alle StopEventResult-Elemente finden
results = root.findall(".//ojp:StopEventResult", ns)

rows = []

for result in results:

    stop_event = result.find("ojp:StopEvent", ns)

    if stop_event is None:
        continue

    # --------------------------------------------------
    # 1. Informationen zum aktuellen Halt
    # --------------------------------------------------

    this_call = stop_event.find(
        "ojp:ThisCall/ojp:CallAtStop",
        ns
    )

    if this_call is None:
        continue

    stop_point_ref = this_call.findtext(
        "siri:StopPointRef",
        default=None,
        namespaces=ns
    )

    stop_name = this_call.findtext(
        "ojp:StopPointName/ojp:Text",
        default=None,
        namespaces=ns
    )

    planned_platform = this_call.findtext(
        "ojp:PlannedQuay/ojp:Text",
        default=None,
        namespaces=ns
    )

    scheduled_departure = this_call.findtext(
        "ojp:ServiceDeparture/ojp:TimetabledTime",
        default=None,
        namespaces=ns
    )

    estimated_departure = this_call.findtext(
        "ojp:ServiceDeparture/ojp:EstimatedTime",
        default=None,
        namespaces=ns
    )

    # --------------------------------------------------
    # 2. Informationen zur Fahrt
    # --------------------------------------------------

    service = stop_event.find(
        "ojp:Service",
        ns
    )

    if service is None:
        continue

    operating_day = service.findtext(
        "ojp:OperatingDayRef",
        default=None,
        namespaces=ns
    )

    journey_ref = service.findtext(
        "ojp:JourneyRef",
        default=None,
        namespaces=ns
    )

    public_code = service.findtext(
        "ojp:PublicCode",
        default=None,
        namespaces=ns
    )

    transport_mode = service.findtext(
        "ojp:Mode/ojp:PtMode",
        default=None,
        namespaces=ns
    )

    product_category = service.findtext(
        "ojp:ProductCategory/ojp:Name/ojp:Text",
        default=None,
        namespaces=ns
    )

    line = service.findtext(
        "ojp:PublishedServiceName/ojp:Text",
        default=None,
        namespaces=ns
    )

    train_number = service.findtext(
        "ojp:TrainNumber",
        default=None,
        namespaces=ns
    )

    origin = service.findtext(
        "ojp:OriginText/ojp:Text",
        default=None,
        namespaces=ns
    )

    destination = service.findtext(
        "ojp:DestinationText/ojp:Text",
        default=None,
        namespaces=ns
    )

    # --------------------------------------------------
    # 3. Als Zeile speichern
    # --------------------------------------------------

    rows.append({
        "operating_day": operating_day,
        "journey_ref": journey_ref,
        "stop_point_ref": stop_point_ref,
        "station_name": stop_name,
        "transport_mode": transport_mode,
        "product_category": product_category,
        "public_code": public_code,
        "line": line,
        "train_number": train_number,
        "origin": origin,
        "destination": destination,
        "planned_platform": planned_platform,
        "scheduled_departure": scheduled_departure,
        "estimated_departure": estimated_departure
    })


# ============================================================
# DATAFRAME ERSTELLEN
# ============================================================

df = pd.DataFrame(rows)

print("Anzahl Verbindungen:", len(df))

display(df)

# Zeitspalten in echte datetime-Werte umwandeln

df["scheduled_departure"] = pd.to_datetime(
    df["scheduled_departure"],
    utc=True,
    errors="coerce"
)

df["estimated_departure"] = pd.to_datetime(
    df["estimated_departure"],
    utc=True,
    errors="coerce"
)


# Schweizer Lokalzeit erzeugen

df["scheduled_departure_local"] = (
    df["scheduled_departure"]
    .dt.tz_convert("Europe/Zurich")
)

df["estimated_departure_local"] = (
    df["estimated_departure"]
    .dt.tz_convert("Europe/Zurich")
)


# Prüfen, ob Echtzeitinformation vorhanden ist

df["has_realtime"] = (
    df["estimated_departure"]
    .notna()
)


# Verspätung berechnen

df["predicted_delay_minutes"] = (
    df["estimated_departure"]
    - df["scheduled_departure"]
).dt.total_seconds() / 60


# Wichtige Spalten anzeigen

display(
    df[
        [
            "station_name",
            "transport_mode",
            "product_category",
            "line",
            "origin",
            "destination",
            "planned_platform",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ]
)



Anzahl Verbindungen: 30


,operating_day,journey_ref,stop_point_ref,station_name,transport_mode,product_category,public_code,line,train_number,origin,destination,planned_platform,scheduled_departure,estimated_departure
0,2026-09-19,ch:1:sjyid:100001:18545-001,ch:1:sloid:3000:503:43,Zürich HB,rail,S-Bahn,S5,S5,18545,Zug,Pfäffikon SZ,43/44,2026-09-19T10:24:00Z,2026-09-19T10:24:18Z
1,2026-09-19,ch:1:sjyid:100001:18844-001,ch:1:sloid:3000:501:33,Zürich HB,rail,S-Bahn,S8,S8,18844,Pfäffikon SZ,Effretikon,33,2026-09-19T10:25:00Z,2026-09-19T10:25:18Z
2,2026-09-19,ch:1:sjyid:100001:18945-001,ch:1:sloid:3000:503:43,Zürich HB,rail,S-Bahn,S9,S9,18945,Rafz,Uster,43/44,2026-09-19T10:28:00Z,2026-09-19T10:28:18Z
3,2026-09-19,ch:1:sjyid:100001:1770-001,ch:1:sloid:3000:9:16,Zürich HB,rail,InterRegio,IR55,IR55,1770,Zürich HB,Biel/Bienne,16,2026-09-19T10:29:00Z,2026-09-19T10:29:24Z
4,2026-09-19,ch:1:sjyid:100001:19146-001,ch:1:sloid:3000:502:42,Zürich HB,rail,S-Bahn,S11,S11,19146,Zürich HB,Aarau,41/42,2026-09-19T10:29:00Z,2026-09-19T10:30:06Z
5,2026-09-19,ch:1:sjyid:100001:18645-001,ch:1:sloid:3000:503:43,Zürich HB,rail,S-Bahn,S6,S6,18645,Baden,Uetikon,43/44,2026-09-19T10:30:00Z,2026-09-19T10:30:30Z
6,2026-09-19,ch:1:sjyid:100001:18646-001,ch:1:sloid:3000:502:42,Zürich HB,rail,S-Bahn,S6,S6,18646,Uetikon,Baden,41/42,2026-09-19T10:31:00Z,2026-09-19T10:31:42Z
7,2026-09-19,ch:1:sjyid:100001:718-001,ch:1:sloid:3000:10:18,Zürich HB,rail,InterCity,IC1,IC1,718,Zürich HB,Genève-Aéroport,18,2026-09-19T10:32:00Z,2026-09-19T10:32:30Z
8,2026-09-19,ch:1:sjyid:100001:151-001,ch:1:sloid:3000:6:11,Zürich HB,rail,EuroCity,EC,EC,151,Basel SBB,Milano Centrale,11,2026-09-19T10:33:00Z,NaN
9,2026-09-19,ch:1:sjyid:100001:568-001,ch:1:sloid:3000:7:12,Zürich HB,rail,InterCity,IC3,IC3,568,Chur,Basel SBB,12,2026-09-19T10:34:00Z,2026-09-19T10:34:30Z


,station_name,transport_mode,product_category,line,origin,destination,planned_platform,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,Zürich HB,rail,S-Bahn,S5,Zug,Pfäffikon SZ,43/44,2026-09-19 12:24:00+02:00,2026-09-19 12:24:18+02:00,0.3
1,Zürich HB,rail,S-Bahn,S8,Pfäffikon SZ,Effretikon,33,2026-09-19 12:25:00+02:00,2026-09-19 12:25:18+02:00,0.3
2,Zürich HB,rail,S-Bahn,S9,Rafz,Uster,43/44,2026-09-19 12:28:00+02:00,2026-09-19 12:28:18+02:00,0.3
3,Zürich HB,rail,InterRegio,IR55,Zürich HB,Biel/Bienne,16,2026-09-19 12:29:00+02:00,2026-09-19 12:29:24+02:00,0.4
4,Zürich HB,rail,S-Bahn,S11,Zürich HB,Aarau,41/42,2026-09-19 12:29:00+02:00,2026-09-19 12:30:06+02:00,1.1
5,Zürich HB,rail,S-Bahn,S6,Baden,Uetikon,43/44,2026-09-19 12:30:00+02:00,2026-09-19 12:30:30+02:00,0.5
6,Zürich HB,rail,S-Bahn,S6,Uetikon,Baden,41/42,2026-09-19 12:31:00+02:00,2026-09-19 12:31:42+02:00,0.7
7,Zürich HB,rail,InterCity,IC1,Zürich HB,Genève-Aéroport,18,2026-09-19 12:32:00+02:00,2026-09-19 12:32:30+02:00,0.5
8,Zürich HB,rail,EuroCity,EC,Basel SBB,Milano Centrale,11,2026-09-19 12:33:00+02:00,NaT,NaN
9,Zürich HB,rail,InterCity,IC3,Chur,Basel SBB,12,2026-09-19 12:34:00+02:00,2026-09-19 12:34:30+02:00,0.5


In [18]:
from pathlib import Path
import pandas as pd

# Zeitpunkt dieses API-Abrufs speichern
collection_time = pd.Timestamp.now(tz="UTC")

df["collection_timestamp"] = collection_time

df["collection_timestamp_local"] = (
    df["collection_timestamp"]
    .dt.tz_convert("Europe/Zurich")
)

# Ordner erstellen, falls er noch nicht existiert
snapshot_dir = Path("data/interim")
snapshot_dir.mkdir(parents=True, exist_ok=True)

# Eindeutiger Dateiname
snapshot_filename = (
    snapshot_dir
    / f"zuerich_hb_{collection_time.strftime('%Y%m%d_%H%M%S')}.csv"
)

# Snapshot speichern
df.to_csv(
    snapshot_filename,
    index=False
)

print("Snapshot gespeichert:")
print(snapshot_filename)

display(
    df[
        [
            "collection_timestamp_local",
            "journey_ref",
            "line",
            "destination",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ]
)

Snapshot gespeichert:
data/interim/zuerich_hb_20260919_102350.csv


,collection_timestamp_local,journey_ref,line,destination,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:18545-001,S5,Pfäffikon SZ,2026-09-19 12:24:00+02:00,2026-09-19 12:24:18+02:00,0.3
1,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:18844-001,S8,Effretikon,2026-09-19 12:25:00+02:00,2026-09-19 12:25:18+02:00,0.3
2,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:18945-001,S9,Uster,2026-09-19 12:28:00+02:00,2026-09-19 12:28:18+02:00,0.3
3,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:1770-001,IR55,Biel/Bienne,2026-09-19 12:29:00+02:00,2026-09-19 12:29:24+02:00,0.4
4,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:19146-001,S11,Aarau,2026-09-19 12:29:00+02:00,2026-09-19 12:30:06+02:00,1.1
5,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:18645-001,S6,Uetikon,2026-09-19 12:30:00+02:00,2026-09-19 12:30:30+02:00,0.5
6,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:18646-001,S6,Baden,2026-09-19 12:31:00+02:00,2026-09-19 12:31:42+02:00,0.7
7,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:718-001,IC1,Genève-Aéroport,2026-09-19 12:32:00+02:00,2026-09-19 12:32:30+02:00,0.5
8,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:151-001,EC,Milano Centrale,2026-09-19 12:33:00+02:00,NaT,NaN
9,2026-09-19 12:23:50.283095+02:00,ch:1:sjyid:100001:568-001,IC3,Basel SBB,2026-09-19 12:34:00+02:00,2026-09-19 12:34:30+02:00,0.5


In [19]:
from pathlib import Path
import pandas as pd

snapshot_files = sorted(
    Path("data/interim").glob("zuerich_hb_*.csv")
)

print("Gefundene Snapshots:", len(snapshot_files))

if len(snapshot_files) < 2:
    print("Noch mindestens einen zweiten Snapshot erstellen.")

else:

    old_file = snapshot_files[-2]
    new_file = snapshot_files[-1]

    print("Alter Snapshot:", old_file.name)
    print("Neuer Snapshot:", new_file.name)

    old_df = pd.read_csv(old_file)
    new_df = pd.read_csv(new_file)

    # Dieselbe Fahrt anhand eindeutiger Merkmale erkennen
    keys = [
        "journey_ref",
        "operating_day",
        "station_name",
        "scheduled_departure"
    ]

    comparison = old_df.merge(
        new_df,
        on=keys,
        how="inner",
        suffixes=("_old", "_new")
    )

    comparison["delay_change_minutes"] = (
        comparison["predicted_delay_minutes_new"]
        - comparison["predicted_delay_minutes_old"]
    )

    print(
        "Fahrten, die in beiden Snapshots vorkommen:",
        len(comparison)
    )

    display(
        comparison[
            [
                "line_old",
                "destination_old",
                "scheduled_departure",
                "predicted_delay_minutes_old",
                "predicted_delay_minutes_new",
                "delay_change_minutes"
            ]
        ]
    )

Gefundene Snapshots: 3
Alter Snapshot: zuerich_hb_20260919_102123.csv
Neuer Snapshot: zuerich_hb_20260919_102350.csv
Fahrten, die in beiden Snapshots vorkommen: 8


,line_old,destination_old,scheduled_departure,predicted_delay_minutes_old,predicted_delay_minutes_new,delay_change_minutes
0,S5,Pfäffikon SZ,2026-09-19 10:24:00+00:00,0.3,0.3,0.0
1,S8,Effretikon,2026-09-19 10:25:00+00:00,0.3,0.3,0.0
2,S9,Uster,2026-09-19 10:28:00+00:00,0.3,0.3,0.0
3,IR55,Biel/Bienne,2026-09-19 10:29:00+00:00,0.4,0.4,0.0
4,S11,Aarau,2026-09-19 10:29:00+00:00,1.1,1.1,0.0
5,S6,Uetikon,2026-09-19 10:30:00+00:00,0.5,0.5,0.0
6,S6,Baden,2026-09-19 10:31:00+00:00,0.7,0.7,0.0
7,IC1,Genève-Aéroport,2026-09-19 10:32:00+00:00,0.5,0.5,0.0


In [20]:
from pathlib import Path
import pandas as pd

snapshot_files = sorted(
    Path("data/interim").glob("zuerich_hb_*.csv")
)

old_df = pd.read_csv(snapshot_files[-2])
new_df = pd.read_csv(snapshot_files[-1])

print("ALTER SNAPSHOT")
print("Anzahl:", len(old_df))
print(
    "Abfahrten von:",
    old_df["scheduled_departure"].min(),
    "bis:",
    old_df["scheduled_departure"].max()
)

print("\nNEUER SNAPSHOT")
print("Anzahl:", len(new_df))
print(
    "Abfahrten von:",
    new_df["scheduled_departure"].min(),
    "bis:",
    new_df["scheduled_departure"].max()
)

old_journeys = set(old_df["journey_ref"].dropna())
new_journeys = set(new_df["journey_ref"].dropna())

common_journeys = old_journeys.intersection(new_journeys)

print("\nGemeinsame journey_ref:")
print(len(common_journeys))

print("\nJourneyRefs alter Snapshot:")
print(old_df[["line", "scheduled_departure", "journey_ref"]])

print("\nJourneyRefs neuer Snapshot:")
print(new_df[["line", "scheduled_departure", "journey_ref"]])

ALTER SNAPSHOT
Anzahl: 10
Abfahrten von: 2026-09-19 10:21:00+00:00 bis: 2026-09-19 10:32:00+00:00

NEUER SNAPSHOT
Anzahl: 30
Abfahrten von: 2026-09-19 10:24:00+00:00 bis: 2026-09-19 10:46:00+00:00

Gemeinsame journey_ref:
8

JourneyRefs alter Snapshot:
   line        scheduled_departure                  journey_ref
0   S24  2026-09-19 10:21:00+00:00  ch:1:sjyid:100001:20445-001
1   S15  2026-09-19 10:22:00+00:00  ch:1:sjyid:100001:19544-001
2    S5  2026-09-19 10:24:00+00:00  ch:1:sjyid:100001:18545-001
3    S8  2026-09-19 10:25:00+00:00  ch:1:sjyid:100001:18844-001
4    S9  2026-09-19 10:28:00+00:00  ch:1:sjyid:100001:18945-001
5  IR55  2026-09-19 10:29:00+00:00   ch:1:sjyid:100001:1770-001
6   S11  2026-09-19 10:29:00+00:00  ch:1:sjyid:100001:19146-001
7    S6  2026-09-19 10:30:00+00:00  ch:1:sjyid:100001:18645-001
8    S6  2026-09-19 10:31:00+00:00  ch:1:sjyid:100001:18646-001
9   IC1  2026-09-19 10:32:00+00:00    ch:1:sjyid:100001:718-001

JourneyRefs neuer Snapshot:
    line      

In [21]:
# ============================================================
# SNAPSHOT COMPARISON
# ============================================================

keys = [
    "operating_day",
    "journey_ref"
]

comparison = old_df.merge(
    new_df,
    on=keys,
    how="inner",
    suffixes=("_old", "_new")
)

comparison["delay_change_minutes"] = (
    comparison["predicted_delay_minutes_new"]
    - comparison["predicted_delay_minutes_old"]
)

print(
    "Fahrten in beiden Snapshots:",
    len(comparison)
)

display(
    comparison[
        [
            "line_old",
            "destination_old",
            "scheduled_departure_old",
            "predicted_delay_minutes_old",
            "predicted_delay_minutes_new",
            "delay_change_minutes"
        ]
    ]
)

Fahrten in beiden Snapshots: 8


,line_old,destination_old,scheduled_departure_old,predicted_delay_minutes_old,predicted_delay_minutes_new,delay_change_minutes
0,S5,Pfäffikon SZ,2026-09-19 10:24:00+00:00,0.3,0.3,0.0
1,S8,Effretikon,2026-09-19 10:25:00+00:00,0.3,0.3,0.0
2,S9,Uster,2026-09-19 10:28:00+00:00,0.3,0.3,0.0
3,IR55,Biel/Bienne,2026-09-19 10:29:00+00:00,0.4,0.4,0.0
4,S11,Aarau,2026-09-19 10:29:00+00:00,1.1,1.1,0.0
5,S6,Uetikon,2026-09-19 10:30:00+00:00,0.5,0.5,0.0
6,S6,Baden,2026-09-19 10:31:00+00:00,0.7,0.7,0.0
7,IC1,Genève-Aéroport,2026-09-19 10:32:00+00:00,0.5,0.5,0.0
